In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from the_well.data import WellDataset
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from modules import *

In [2]:
SEED = 42
EPOCHS = 100
BATCH_SIZE = 1

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

Dispositivo: cuda
Torch: 2.11.0+cu128  |  Torchvision: 0.26.0+cu128


In [4]:
train_dataset = WellDataset(
    path="/mnt/storage_C1/BILL_pino/the_well/datasets/helmholtz_staircase",
    well_split_name="train"
)

validation_dataset = WellDataset(
    path="/mnt/storage_C1/BILL_pino/the_well/datasets/helmholtz_staircase",
    well_split_name="valid"
)

'''
sample = train_dataset[0]

for k, v in sample.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))
'''

'\nsample = train_dataset[0]\n\nfor k, v in sample.items():\n    if torch.is_tensor(v):\n        print(k, v.shape, v.dtype)\n    else:\n        print(k, type(v))\n'

In [5]:
class HelmholtzDataset(torch.utils.data.Dataset):
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]

        x = torch.cat([
            sample["input_fields"][0],
            sample["constant_fields"],
            sample["space_grid"]
        ], dim=-1)

        y = sample["output_fields"][0]

        return x.float(), y.float()

In [6]:
train_ds = HelmholtzDataset(train_dataset)
val_ds = HelmholtzDataset(validation_dataset)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [7]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d(
    modes1=16,
    modes2=16,
    width=32,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=4
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

criterion = relative_l2_loss

/home/al.igor.zwirtes/Documentos/AV2_ML2_IMPA/.env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


In [8]:
train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS
)

Epoch 1/100:  22%|██▏       | 4491/20384 [02:17<08:07, 32.61it/s, loss=0.161642]


KeyboardInterrupt: 